<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/create_nemo_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Read in csv of paths to audio files and create a manifest that can be used in NeMO

The main thing NeMO does is add in duration into the manifest.

In [ ]:
#@title Imports and constants
import pandas as pd
import torch
import torchaudio
from nemo.collections.asr.parts.utils import manifest_utils

csv_path = '/usr/local/google/home/varungulshan/radio-transcription/model/data/manifests/s3_samples.csv'  #@param
output_manifest = '/usr/local/google/home/varungulshan/radio-transcription/model/data/manifests/s3_samples.json'  #@param

In [ ]:
def create_nemo_manifest_from_urls(csv_path, output_manifest):
    df = pd.read_csv(csv_path)
    manifest_entries = []

    print(f"Starting manifest creation for {len(df)} entries...")

    for i, row in df.iterrows():
        url = row.iloc[0]

        try:
            # NeMo's internal data loaders use torchaudio.load()
            # for URLs when the ffmpeg backend is active.
            # We load the file to get the exact duration NeMo will 'see' at runtime.
            waveform, sample_rate = torchaudio.load(url)

            # Duration = number of samples / sample rate
            duration = waveform.shape[1] / sample_rate

            # Build the manifest entry with NeMo-required fields
            entry = {
                "audio_filepath": url,
                "duration": round(float(duration), 3),
                "text": row.get('text', "fake transcription"),  # NeMo requires 'text' for most tasks
                "lang": "en",  # transcribe_speech.py reads this field off to configure models
            }
            manifest_entries.append(entry)
            print(f"[{i+1}/{len(df)}] Success: {url}")

        except Exception as e:
            print(f"[{i+1}/{len(df)}] Failed: {url} | Error: {e}")

    # Use NeMo's official utility to write the manifest
    # This ensures the correct JSONL format (one line per entry, no commas)
    manifest_utils.write_manifest(output_manifest, manifest_entries)
    print(f"\nManifest successfully saved to: {output_manifest}")

# Usage
create_nemo_manifest_from_urls(csv_path, output_manifest)